# Task 06 - Kaggle GRU training and finalization record

This curated notebook is derived from the original downloaded Kaggle notebook. Genuine outputs are retained only for unchanged execution cells. One-off checkpoint debugging, source inspection, patch discovery/upload, and redundant recovery cells were removed. The unmodified downloaded notebook remains outside the Git repository in the parent workspace.


In [ ]:
from pathlib import Path
import shutil
import sys

# Locate the single uploaded project directory without depending on an account name.
project_name = "Capstone-Project_268513X"
matches = sorted(path for path in Path("/kaggle/input").rglob(project_name) if path.is_dir())
assert len(matches) == 1, f"Expected one uploaded project, found: {matches}"
source = matches[0]
project = Path("/kaggle/working") / project_name

if not project.exists():
    shutil.copytree(source, project)
    print("Project copied successfully.")
else:
    print("Project already exists; not overwriting.")

print("Working directory:", project)
print("Python:", sys.version.split()[0])
print("GRU script ready:", (project / "scripts/run_gru.py").is_file())


In [2]:
import torch
import numpy
import pandas
import scipy
import sklearn
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU unavailable! Enable GPU T4 x2."

print("GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

required = [
    "data/raw/metr-la/METR-LA.csv",
    "data/raw/metr-la/graph_sensor_ids.txt",
    "artifacts/task04_baselines/test_predictions_horizons.npz",
    "artifacts/task05_xgboost/test_predictions_horizons.npz",
    "scripts/run_gru.py",
    "src/models/gru.py",
    "src/training/gru.py",
]

for name in required:
    path = project / name
    assert path.is_file(), f"Missing file: {name}"
    print("OK:", name)

result = subprocess.run(
    [sys.executable, "scripts/run_gru.py", "--help"],
    cwd=str(project),
    capture_output=True,
    text=True,
    check=True,
)

assert "--probe-only" in result.stdout
assert "--full" in result.stdout

print("\nAll checks passed. Ready for full GRU training.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU count: 2
GPU memory: 14.56 GB
OK: data/raw/metr-la/METR-LA.csv
OK: data/raw/metr-la/graph_sensor_ids.txt
OK: artifacts/task04_baselines/test_predictions_horizons.npz
OK: artifacts/task05_xgboost/test_predictions_horizons.npz
OK: scripts/run_gru.py
OK: src/models/gru.py
OK: src/training/gru.py

All checks passed. Ready for full GRU training.


## Original full-training workflow

The command below is the exact command used for both approved candidates: batch size 4096, two workers, CUDA, and 20 throughput-probe batches. Its authentic saved output is retained because it shows that both candidates completed before the finalization RNG-state exception. The traceback is historical execution evidence and is resolved by the later finalize-only workflow; the training command is not re-executed in this curated notebook.


In [3]:
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "scripts/run_gru.py",
    "--full",
    "--device", "cuda",
    "--batch-size", "4096",
    "--num-workers", "2",
    "--throughput-probe-batches", "20",
]

print("Starting Task 06 full GRU training...", flush=True)

result = subprocess.run(
    command,
    cwd=str(project),
    check=False,
)

print("\nTraining exit code:", result.returncode)
assert result.returncode == 0, "GRU training failed"

print("Task 06 GRU training completed successfully!")

Starting Task 06 full GRU training...
{"epoch": 0, "train_mae_mph": 3.6816553855043193, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.836816240260862, "30": 3.3496802110544244, "60": 4.19302996354288}, "selection_score_mph": 3.459842138286055}
{"epoch": 1, "train_mae_mph": 3.483907031090611, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.8525028882258483, "30": 3.3049547767912033, "60": 4.046840317886519}, "selection_score_mph": 3.4014326609678576}
{"epoch": 2, "train_mae_mph": 3.4203388274363493, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.80677885883887, "30": 3.260476836944079, "60": 3.9991302318334023}, "selection_score_mph": 3.3554619758721174}
{"epoch": 3, "train_mae_mph": 3.384118030472521, "train_eligible_count": 55289280, "validation_horizon_mae_mph": {"15": 2.7919989361907582, "30": 3.2388756142179522, "60": 3.965972267583912}, "selection_score_mph": 3.3322822726642074}
{"epoch": 4, "train_mae_m

Traceback (most recent call last):
  File "/kaggle/working/Capstone-Project_268513X/scripts/run_gru.py", line 516, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/Capstone-Project_268513X/scripts/run_gru.py", line 507, in main
    summary["selected"] = finalize_full_run(
                          ~~~~~~~~~~~~~~~~~^
        trials, configs, datasets, windows, frame, device, args.num_workers,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        args.batch_size, data_path, splits,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/kaggle/working/Capstone-Project_268513X/scripts/run_gru.py", line 383, in finalize_full_run
    checkpoint = load_checkpoint(
        ROOT / selected["best_checkpoint"], model=model, optimizer=optimizer,
        training_config=config, data_generator=generator, map_location=device,
    )
  File "/kaggle/working/Capstone-Project_268513X/src/training/gru.py", line 225, in load_checkpoint
    restore_rng_state


Training exit code: 1


AssertionError: GRU training failed

### Recorded training result

Both candidates completed successfully before finalization failed. Validation selected `gru_h128` (70,649 parameters) at epoch 19 with a mean +15/+30/+60 validation MAE of 3.2292559775 mph. `gru_h64` achieved 3.2469073768 mph. The failure occurred only while restoring CUDA-mapped CPU RNG state during finalization; no retraining was required.


## Optional finalize-only recovery

The following successful output is retained from the original Kaggle execution after the local source recovery fix was applied. Finalize-only reused completed validation evidence and best checkpoints; it did not train or rerun model selection. The patch-upload and checkpoint-debugging cells are intentionally omitted.


In [ ]:
REPO = project
print("Recovery repository:", REPO)


In [12]:
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "scripts/run_gru.py",
    "--finalize-only",
    "--device", "cuda",
    "--batch-size", "4096",
    "--num-workers", "2",
]

print("Starting Task 06 finalization...", flush=True)

result = subprocess.run(
    command,
    cwd=str(REPO),
    check=False,
)

print("\nExit code:", result.returncode)
assert result.returncode == 0, "Finalization failed"

print("Task 06 finalization completed successfully!")

Starting Task 06 finalization...
{
  "mode": "finalize-only",
  "selected": {
    "candidate": "gru_h128",
    "training_config": {
      "candidate_name": "gru_h128",
      "model": {
        "history_features": 3,
        "hidden_size": 128,
        "sensor_count": 207,
        "sensor_embedding_dim": 8,
        "output_length": 12,
        "future_time_features": 2
      },
      "learning_rate": 0.001,
      "batch_size": 4096,
      "max_epochs": 40,
      "patience": 6,
      "gradient_clip_norm": 1.0,
      "seed": 268513
    },
    "parameter_count": 70649,
    "best_epoch": 19,
    "best_validation_score_mph": 3.2292559775068383,
    "checkpoint": "artifacts/task06_gru/checkpoints/gru_h128_best.pt",
    "checkpoint_sha256": "4FC1A1A86EED8BABC5F5C5A8B33EEA86CD8086399A03D2795723EAC2E9343D28"
  }
}

Exit code: 0
Task 06 finalization completed successfully!


## Completed artifact verification and packaging

These unchanged cells and their genuine outputs confirm the final metrics/configuration files and the downloadable Task 06 artifact archive produced by the recovered run.


In [13]:
import json
from pathlib import Path

repo = Path("/kaggle/working/Capstone-Project_268513X")

metrics_path = repo / "results/task06_gru/metrics.json"
predictions_path = repo / "artifacts/task06_gru/test_predictions_horizons.npz"
config_path = repo / "artifacts/task06_gru/selected_config.json"

for path in [metrics_path, predictions_path, config_path]:
    assert path.is_file(), f"Missing output: {path}"
    print("OK:", path.relative_to(repo))

print("\n=== TASK 06 TEST METRICS ===")
with metrics_path.open(encoding="utf-8") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

OK: results/task06_gru/metrics.json
OK: artifacts/task06_gru/test_predictions_horizons.npz
OK: artifacts/task06_gru/selected_config.json

=== TASK 06 TEST METRICS ===
{
  "task": "Task 06 - Sensor-local GRU",
  "created_utc": "2026-11-11T11:01:39.668149+00:00",
  "data_path": "data/raw/metr-la/METR-LA.csv",
  "data_sha256": "8D67A35472DB1719D7D4BE851F2BF64CB21D9C52577C8A6B4B873D43205AF381",
  "split_policy": "chronological_70_10_20",
  "splits": {
    "train": {
      "start_row": 0,
      "stop_row_exclusive": 23990
    },
    "validation": {
      "start_row": 23990,
      "stop_row_exclusive": 27417
    },
    "test": {
      "start_row": 27417,
      "stop_row_exclusive": 34272
    }
  },
  "window_policy": "12 sensor-local input steps to 12 jointly trained target steps",
  "feature_policy": "local standardized speed, zero/finite indicators, target time-of-day, learned sensor identity; no graph or neighbouring speeds",
  "selection_policy": "lowest equal-weight mean validation MAE 

In [14]:
from pathlib import Path
import zipfile

repo = Path("/kaggle/working/Capstone-Project_268513X")
output = Path("/kaggle/working/UrbanFlow_AI_Task06_Complete.zip")

with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    for directory in [
        "artifacts/task06_gru",
        "results/task06_gru",
    ]:
        for path in (repo / directory).rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(repo))

print("Backup created:", output)
print("Size:", round(output.stat().st_size / 1024**2, 2), "MB")

Backup created: /kaggle/working/UrbanFlow_AI_Task06_Complete.zip
Size: 17.74 MB
